# B-10: recursive-rollout improvements (does anything fix the spike-blindness?)

B-09 (D-53) established that recursive daily rollout does not catastrophically compound error over
365 days, but R2 stayed poor for most models (multi-anchor mean: XGB best at 0.003) even though MASE
mostly beat persistence -- the classic "MASE<1 alongside near-zero/negative R2 = spike-tail
signature" (D-44b). This notebook tests three concrete improvement ideas, all reusing
`recursive_rollout.py`'s existing B-09 machinery (extended, not replaced):

1. **Blended AR** -- blend each tree model's recursive memory with a day-of-year climatology value
   (`alpha*pred + (1-alpha)*climatology` fed back into history), testing whether pure recursion is
   actually adding value over just anchoring to climatology. alpha in {1.0, 0.5, 0.0}.
2. **Ensemble** -- mean (unweighted and MASE-weighted, weights frozen from B-09's own multi-anchor
   MASE, not re-derived here) of RF+XGB+LightGBM+SARIMAX.
3. **H=1 DL retrain** -- retrain DLinear/LSTM natively for single-step-ahead (`H=1`) instead of
   reusing the existing H=14-trained model's first output, in case that horizon mismatch was why
   DLinear especially was unstable in B-09.

**Design, unchanged from B-09**: single fixed anchor date (2021-12-16), Tower 4, real
perfect-foresight `fx_` drivers throughout, lead-time-binned evaluation
(`recursive_rollout.bin_metrics`) against real `y_observed`. **This notebook formalizes the
single-anchor smoke test only** -- per B-09's own headline lesson ("don't trust a single anchor"),
the actual improvement verdicts below are drawn from a 5-anchor (2018-2022) sweep run as a
script extension (`b10_multi_anchor.py`, mirroring B-09's `b09_multi_anchor.py` precedent of not
re-executing the multi-anchor loop inside the notebook itself) -- results in `b10_results.md`.

In [1]:
from pathlib import Path
import sys, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.insert(0, "../../src")

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

import models.forecasting_dl as fdl
import models.recursive_rollout as rr

HOURLY = Path("../../data/Hourly"); RESULTS = Path("../../results")
ANCHOR = pd.Timestamp("2021-12-16")
N_DAYS = 365
TOWER = 4
DUM = ["is_t2", "is_t4", "is_t9"]
AR_COLS = ["ar_ch4_dlag1", "ar_ch4_dlag2", "ar_ch4_dlag3", "ar_ch4_dlag7", "ar_ch4_dlag14", "ar_ch4_drm7"]
EXOG_B = ["fx_lsu_dens", "fx_WS_mean", "fx_VPD_mean", "fx_USTAR_mean", "fx_PPFD_mean",
          "fx_DOY_sin", "fx_DOY_cos", "fx_is_growing"]
# frozen B-09 multi-anchor mean MASE (results/b09_multi_anchor_summary.csv), NOT re-derived here
B09_MEAN_MASE = {"XGB": 0.968, "LightGBM": 0.978, "RF": 1.024, "SARIMAX": 1.038}
print(f"Anchor: {ANCHOR.date()}  ->  target window {(ANCHOR+pd.Timedelta(days=1)).date()} .. {(ANCHOR+pd.Timedelta(days=N_DAYS)).date()}")

Anchor: 2021-12-16  ->  target window 2021-12-17 .. 2022-12-16


## 1  Load data

In [2]:
dv = pd.read_csv(HOURLY/"forecast_daily_v2.csv", low_memory=False)
dv["Datetime"] = pd.to_datetime(dv["Datetime"], format="mixed")
FX_B = [c for c in dv.columns if c.startswith("fx")]
T = {t: dv[dv.tower == t].set_index("Datetime").sort_index() for t in [2, 4, 9]}

feat_cols = AR_COLS + FX_B + ["ar_fc_dlag1"] + DUM
target_dates = pd.date_range(ANCHOR + pd.Timedelta(days=1), periods=N_DAYS, freq="D")
df4 = T[TOWER]
history_init = df4.loc[:ANCHOR, "y_gapfilled"].copy()
fx_frame = df4.loc[target_dates, FX_B + ["ar_fc_dlag1"]].copy()
fx_frame["is_t2"], fx_frame["is_t4"], fx_frame["is_t9"] = 0.0, 1.0, 0.0
print(f"Pre-anchor real history: {len(df4.loc[:ANCHOR])} rows")

Pre-anchor real history: 1811 rows


## 2  Part 1 -- blended AR (RF/XGB/LightGBM only; SARIMAX has no equivalent AR column --
its memory is its own Kalman state, not scoped here)

`recursive_rollout.tree_rollout` extended with optional, default-preserving `alpha`/`clim_series`
params (D-54) -- `alpha=1.0, clim_series=None` reproduces B-09's exact original chain bit-for-bit
(verified below). When given, blending happens **at append time**: what gets fed back into memory
for day d is `alpha*pred + (1-alpha)*clim_series[d]` -- the reported/evaluated prediction for day d
itself is always the model's own raw `pred`, never the blended value.

In [3]:
def fit_tree(algo, tr, feat_cols):
    imp = SimpleImputer(strategy="mean"); Xi = imp.fit_transform(tr[feat_cols].values)
    if algo == "RF":
        m = RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=42,
                                   min_samples_leaf=10, max_features=0.5)
    elif algo == "XGB":
        m = XGBRegressor(subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42,
                          max_depth=2, learning_rate=0.02, n_estimators=400, min_child_weight=10)
    elif algo == "LightGBM":
        m = LGBMRegressor(subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42,
                           num_leaves=7, min_child_samples=10, learning_rate=0.02, n_estimators=400,
                           verbosity=-1)
    else:
        raise ValueError(algo)
    m.fit(Xi, tr["target"].values); return m, imp

pool = []
for t in [2, 4, 9]:
    df = T[t].copy(); df["target"] = df["y_gapfilled"]
    for d in DUM: df[d] = 1.0 if d == f"is_t{t}" else 0.0
    pool.append(df[df.index <= ANCHOR])
tr = pd.concat(pool); tr = tr[tr["target"].notna()]
print(f"Pooled training rows (<= anchor): {len(tr)}")

hist_obs = df4.loc[:ANCHOR - pd.Timedelta(days=1), "y_observed"].dropna()
clim_vals = rr.doy_climatology(hist_obs, target_dates, window=7)
clim_series = pd.Series(clim_vals, index=target_dates)

blend_chains = {}
for algo in ["RF", "XGB", "LightGBM"]:
    t0 = time.time()
    model, imp = fit_tree(algo, tr, feat_cols)
    for alpha in [1.0, 0.5, 0.0]:
        blend_chains[(algo, alpha)] = rr.tree_rollout(model, imp, feat_cols, fx_frame, history_init,
                                                        ANCHOR, n_days=N_DAYS, alpha=alpha, clim_series=clim_series)
    print(f"  {algo} (3 alphas) done ({time.time()-t0:.0f}s)", flush=True)

# backward-compat check: alpha=1.0 must reproduce the pure-recursive chain exactly
b09_chains_check = pd.read_csv(RESULTS/"b09_chains.csv", index_col=0, parse_dates=True)
for algo in ["RF", "XGB", "LightGBM"]:
    diff = (blend_chains[(algo, 1.0)] - b09_chains_check[algo]).abs().max()
    print(f"  {algo} alpha=1.0 vs B-09 original chain: max abs diff = {diff:.10f} (expect 0)")

Pooled training rows (<= anchor): 5433


  RF (3 alphas) done (48s)


  XGB (3 alphas) done (2s)


  LightGBM (3 alphas) done (2s)


  RF alpha=1.0 vs B-09 original chain: max abs diff = 0.0000000000 (expect 0)
  XGB alpha=1.0 vs B-09 original chain: max abs diff = 0.0000000000 (expect 0)
  LightGBM alpha=1.0 vs B-09 original chain: max abs diff = 0.0000000000 (expect 0)


## 3  SARIMAX (reused for the ensemble in Part 2)

In [4]:
y = df4["y_gapfilled"].astype(float)
X = df4[EXOG_B].astype(float).ffill().bfill()
y_tr, X_tr = y.loc[:ANCHOR], X.loc[:ANCHOR]

best = None
for p in [1, 2]:
    for q in [0, 1]:
        try:
            m = SARIMAX(y_tr, exog=X_tr, order=(p, 1, q), enforce_stationarity=False, enforce_invertibility=False)
            res = m.fit(disp=False, maxiter=50)
            if best is None or res.aic < best[0]: best = (res.aic, (p, 1, q), res)
        except Exception:
            continue
sarimax_order, sarimax_res = best[1], best[2]
future_X = X.loc[target_dates]
fc = sarimax_res.get_forecast(steps=N_DAYS, exog=future_X)
sarimax_chain = pd.Series(fc.predicted_mean.values, index=target_dates)
print(f"SARIMAX order={sarimax_order}")

C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


SARIMAX order=(2, 1, 1)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


## 4  Part 2 -- ensemble

`Ensemble_unweighted` = simple mean of RF+XGB+LightGBM+SARIMAX (alpha=1.0 chains).
`Ensemble_MASEweighted` = weighted mean, `weight_i = (1/MASE_i) / sum_j(1/MASE_j)` using
**B-09's frozen multi-anchor mean MASE** (not derived from this notebook's own evaluation window --
that would be circular).

In [5]:
tree_a1 = {algo: blend_chains[(algo, 1.0)] for algo in ["RF", "XGB", "LightGBM"]}
ens_df = pd.DataFrame({**tree_a1, "SARIMAX": sarimax_chain})
ens_unweighted = ens_df.mean(axis=1)
w = {k: (1.0 / v) for k, v in B09_MEAN_MASE.items()}
wsum = sum(w.values()); w = {k: v / wsum for k, v in w.items()}
ens_weighted = sum(ens_df[k] * w[k] for k in ens_df.columns)
print("ensemble weights (MASE-weighted):", {k: round(v, 4) for k, v in w.items()})

ensemble weights (MASE-weighted): {'XGB': 0.2586, 'LightGBM': 0.2559, 'RF': 0.2444, 'SARIMAX': 0.2411}


## 5  Part 3 -- H=1 DL retrain

`forecasting_dl.make_windows(ser, L, H, stride=1)` already accepts `H` as a plain parameter --
`TRACKS`/`build_windows` are what hardcode `H=14` for track B, not `make_windows` itself. Calling it
directly with `H=1` bypasses `TRACKS`/`build_windows` entirely -- zero new shared-module code, no
production DL code path (B-02/B-03b/B-04) touched. `rr.dl_rollout(..., H=1, ...)` then treats
`pred[0]` as the only (not first-of-14) predicted step each day.

In [6]:
device = fdl.get_device(); print("device:", device)
track = "B"
m = fdl.load_matrix(HOURLY/"forecast_features_v2.csv")
cutoff = pd.Timestamp("2021-12-16 23:59")

windows = {t: fdl.make_windows(fdl.tower_series(m, t, track), L=28, H=1, stride=1) for t in [2, 4, 9]}
tr_parts = [fdl._subset(windows[t], pd.DatetimeIndex(windows[t]["ttime"][:, -1]) <= cutoff) for t in [2, 4, 9]]
train = fdl._cat(tr_parts)
print(f"H=1 window shapes: enc={train['enc'].shape}, dec={train['dec'].shape} (expect dec (...,1,F_dec))")
assert train["dec"].shape[1] == 1, "H=1 window construction failed sanity check"

se, sd = fdl.Scaler().fit(train["enc"]), fdl.Scaler().fit(train["dec"])
yv = train["y"][np.isfinite(train["y"])]; mu, sdy = float(yv.mean()), float(yv.std() + 1e-6)
train["enc"], train["dec"] = se.tf(train["enc"]), sd.tf(train["dec"])
n_enc, n_dec = train["enc"].shape[-1], train["dec"].shape[-1]

ser4 = fdl.tower_series(m, TOWER, track)
dates_full, enc_ex_full, dec_ex_full = ser4["idx"], ser4["enc_ex"], ser4["dec_ex"]
Y_full_real, ch4_full_real = ser4["Y"], ser4["ch4"]
anchor_idx = dates_full.get_loc(ANCHOR)
dl_history_init = ch4_full_real[:anchor_idx + 1]

h1_chains = {}
for name in ["DLinear", "LSTM"]:
    t0 = time.time()
    model = fdl.build_model(name, 28, 1, n_enc, n_dec, 3)
    fdl.train_model(model, train, device, epochs=30, ch4_mu=mu, ch4_sd=sdy, seed=0)
    h1_chains[name] = rr.dl_rollout(model, se, sd, mu, sdy, device, fdl.TOW[TOWER],
                                     enc_ex_full, dec_ex_full, dates_full, dl_history_init, ANCHOR,
                                     L=28, H=1, n_days=N_DAYS)
    print(f"  {name}-H1 rollout done ({time.time()-t0:.0f}s)", flush=True)

device: cuda


H=1 window shapes: enc=(5349, 28, 30), dec=(5349, 1, 26) (expect dec (...,1,F_dec))


  DLinear-H1 rollout done (2s)


  LSTM-H1 rollout done (2s)


## 6  Evaluation -- single-anchor (2021-12-16) results for all three parts

In [7]:
y_true_full = pd.Series(Y_full_real, index=dates_full)
y_true = y_true_full.reindex(target_dates).values
anchor_val = df4.loc[ANCHOR, "y_gapfilled"]
persist = rr.chain_persistence(anchor_val, N_DAYS)

rows = []
for (algo, alpha), chain in blend_chains.items():
    yp = chain.reindex(target_dates).values
    bm = rr.bin_metrics(y_true, yp, target_dates, ANCHOR, y_persist=persist)
    bm["model"] = algo; bm["alpha"] = alpha
    rows.append(bm)
R1 = pd.concat(rows, ignore_index=True)
R1.to_csv(RESULTS/"b10_ar_blend_summary.csv", index=False)

rows = []
for name, chain in [("Ensemble_unweighted", ens_unweighted), ("Ensemble_MASEweighted", ens_weighted),
                    ("SARIMAX", sarimax_chain)] + list(tree_a1.items()):
    yp = chain.reindex(target_dates).values
    bm = rr.bin_metrics(y_true, yp, target_dates, ANCHOR, y_persist=persist)
    bm["model"] = name
    rows.append(bm)
R2 = pd.concat(rows, ignore_index=True)
R2.to_csv(RESULTS/"b10_ensemble_summary.csv", index=False)

rows = []
for name, chain in h1_chains.items():
    yp = chain.reindex(target_dates).values
    bm = rr.bin_metrics(y_true, yp, target_dates, ANCHOR, y_persist=persist)
    bm["model"] = name; bm["h1_retrain"] = True
    rows.append(bm)
R3 = pd.concat(rows, ignore_index=True)
R3.to_csv(RESULTS/"b10_h1_retrain_summary.csv", index=False)

pd.set_option("display.width", 200)
print("=== Part 1: R2 by model x alpha, by bin ===")
print(R1.pivot_table(index=["model", "alpha"], columns="bin", values="R2").round(3).to_string())
print("\n=== Part 2: ensemble vs individuals, R2 by bin ===")
print(R2.pivot_table(index="model", columns="bin", values="R2").round(3).to_string())
print("\n=== Part 3: H=1 retrain, R2 by bin (vs B-09's H=14-truncated originals) ===")
print(R3.pivot_table(index="model", columns="bin", values="R2").round(3).to_string())
rows = []
for name in ["DLinear", "LSTM"]:
    yp = b09_chains_check[name].reindex(target_dates).values
    bm = rr.bin_metrics(y_true, yp, target_dates, ANCHOR, y_persist=persist)
    bm["model"] = name + "_H14orig"
    rows.append(bm)
R3b = pd.concat(rows, ignore_index=True)
print(R3b.pivot_table(index="model", columns="bin", values="R2").round(3).to_string())

=== Part 1: R2 by model x alpha, by bin ===
bin               1-7  181-270  271-365  31-90   8-30  91-180
model    alpha                                               
LightGBM 0.0   -3.872    0.110   -0.156 -0.158  0.096   0.054
         0.5   -0.710    0.215   -0.211 -0.125  0.104   0.117
         1.0   -0.606    0.335   -0.582 -0.056 -0.214   0.193
RF       0.0   -5.343    0.127   -0.181 -0.204 -0.069   0.090
         0.5   -4.827    0.155   -0.246 -0.185 -0.148   0.145
         1.0   -3.127    0.222   -0.594 -0.182 -0.732   0.209
XGB      0.0   -4.858    0.109   -0.146 -0.194  0.095   0.045
         0.5   -2.028    0.211   -0.163 -0.166  0.088   0.100
         1.0   -1.803    0.322   -0.514 -0.098 -0.170   0.137

=== Part 2: ensemble vs individuals, R2 by bin ===
bin                      1-7  181-270  271-365  31-90   8-30  91-180
model                                                               
Ensemble_MASEweighted -2.352    0.294   -0.439 -0.036 -0.201   0.168
Ensemble_unweig

## 7  Multi-anchor (2018-2022) extension -- the actual improvement verdicts

Per B-09's own headline lesson, single-anchor results above are a smoke test only -- run as a
script extension (`b10_multi_anchor.py`, not re-executed here, mirroring `b09_multi_anchor.py`'s
precedent) across the same 5 anchor years B-09 used. Results: `results/b10_ar_blend_multi_anchor.csv`,
`results/b10_ensemble_multi_anchor.csv`, `results/b10_h1_retrain_multi_anchor.csv`. Full
interpretation in `b10_results.md`. Headline (n-weighted mean R2/MASE across 5 anchors):

- **Part 1 (blended AR)**: alpha=1.0 (pure recursive) beats alpha=0.5 and alpha=0.0 for all three
  tree models, monotonically -- blending in climatology only dilutes signal recursion already
  captures. This generalizes the single-anchor finding; **idea fails to improve on B-09**.
- **Part 2 (ensemble)**: `Ensemble_unweighted` R2=0.012, `Ensemble_MASEweighted` R2=0.011 --
  both slightly **beat** the best individual model (XGB, R2=0.003), though MASE ticks up marginally
  (0.975 vs 0.968). A modest, genuine improvement.
- **Part 3 (H=1 retrain)**: mixed. LSTM-H1 improves on both R2 (-0.364 vs -0.438) and MASE (1.073 vs
  1.104) -- a real, if small, win. DLinear-H1 is worse on R2 (-1.729 vs -1.460) despite a slightly
  better MASE (1.542 vs 1.580) -- DLinear's instability (already flagged in B-09 as anchor-dependent
  and not robust) persists under H=1 retraining too.

## 8  Append to benchmarks.csv (B10, single-anchor smoke-test rows)

In [8]:
bench = RESULTS/"benchmarks.csv"; today = pd.Timestamp.today().date().isoformat()
ex = pd.read_csv(bench); ex = ex[ex["replication"] != "B10"]
rows = []
for label, R, extra_note in [
    ("blendAR", R1, "Part 1 blended-AR ablation (alpha in {1.0,0.5,0.0})"),
    ("ensemble", R2, "Part 2 ensemble of RF+XGB+LightGBM+SARIMAX"),
    ("h1retrain", R3, "Part 3 H=1 DL retrain vs B09 H=14-truncated"),
]:
    for _, r in R.iterrows():
        lo, hi = r["bin"].split("-")
        model_label = f"{r['model']}_a{r['alpha']}" if "alpha" in R.columns and pd.notna(r.get("alpha", np.nan)) else r["model"]
        rows.append({"replication": "B10", "model": model_label, "tower": f"Tower {TOWER}",
            "feature_set": f"recursive rollout improvement ({label}); single-anchor smoke test 2021-12-16; see b10_multi_anchor.csv for 5-anchor verdicts",
            "track": "B", "horizon": int(hi), "split": f"b10_{label}_leadtime_bin_{r['bin']}",
            "R2": r["R2"], "MAE": r["MAE"], "n_test": int(r["n"]),
            "MASE": r["MASE"], "date": today,
            "notes": f"{extra_note}; lead-time bin days {r['bin']}; D-54"})
new = pd.DataFrame(rows); comb = pd.concat([ex, new], ignore_index=True); comb.to_csv(bench, index=False)
print(f"Wrote {len(new)} B10 rows. Total {len(comb)}.")

Wrote 102 B10 rows. Total 3887.
